[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Tasks](Tasks.md) | [Task 1](Task-1.md) | [Task 2](Task-2.md) | Notebook

# AS Centrality

In [22]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [23]:
%pip install -q pybgpkit-parser pytricia pandas matplotlib jinja2 requests

from array import array
from collections import defaultdict
import gc
import io
import ipaddress
import math
from pathlib import Path
import time
import requests
import urllib.request
import shutil
import multiprocessing as mp

import jinja2
import matplotlib.pyplot as plt
import pandas as pd
import pybgpkit_parser as bgpkit
import pytricia

Note: you may need to restart the kernel to use updated packages.


## Obtain BGP data

In the [BGP assignment](https://github.com/CAIDA/nids-bgp-control-plane), you fetched BGP Routing Information Base (RIB) snapshots from a *single* collector managed by [RouteViews](https://www.routeviews.org/routeviews/). This time, you will fetch the same type of data from *multiple* collectors, managed by [RIPE RIS](https://www.ripe.net/analyse/internet-measurements/routing-information-service-ris/). 

Using data from more collectors allows for a richer view of the Internet's structure at the cost of requiring significantly more computing power. When first writing your code, you can use `COLLECTORS = ["rrc06"]` so that the notebook executes more quickly. When answering the questions for the tasks, use the larger set of data with `COLLECTORS = ["rrc00", "rrc15", "rrc23"]`.

| Collector | Location | File size |
|---|---|---|
| `rrc00` | Amsterdam, NL | ~404 MB |
| `rrc06` | Otemachi, JP | ~41 MB |
| `rrc15` | São Paolo, BR | ~137 MB |
| `rrc23` | Singapore, SG | ~81 MB |

In [24]:
COLLECTORS = ["rrc06"]                        # Uncomment for testing
# COLLECTORS = ["rrc00", "rrc15", "rrc23"]    # Uncomment for full analysis
SNAPSHOT_DATE = "20260801"                    # YYYYMMDD
SNAPSHOT_HOUR = "0000"                        # 0000, 0800, 1600 UTC

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_rib_file(collector, date=None, hour=None):
    """
    Download a RIB file into data directory if not already cached;
    Return file path.
    """
    date = date or SNAPSHOT_DATE
    hour = hour or SNAPSHOT_HOUR
    rib_name = f"bview.{collector}.{date}.{hour}.gz"
    rib_path = DATA_DIR / f"{rib_name}"
    if rib_path.exists():
        print(f"Found in cache: {rib_name}")
    else:
        rib_path.parent.mkdir(parents=True, exist_ok=True)
        download_url = (
            f"https://data.ris.ripe.net/"
            f"{collector}/{date[:4]}.{date[4:6]}/bview.{date}.{hour}.gz"
        )
        with requests.get(download_url, stream=True) as resp:
            resp.raise_for_status()
            with open(rib_path, "wb") as f_out:
                shutil.copyfileobj(resp.raw, f_out)
        print(f"Downloaded: {rib_name}")
    return rib_path

RIB_PATHS = [ fetch_rib_file(c) for c in COLLECTORS ]
print("Fetched all RIB files.")

Found in cache: bview.rrc06.20260801.0000.gz
Fetched all RIB files.


# Convert RIB data into a weighted graph

Conceptually, BC and hegemony treat ASes and AS paths as vertices and edges in a graph. In this task, you will process the RIB files that you just fetched, and extract AS path information for later use. 

You might want to refer to the [bgpkit documentation](https://docs.rs/bgpkit-parser/latest/bgpkit_parser).

`TODO`: if two peers are seeing the same AS path, then we might not want to double count them. 

## Compute Address Space Size

This function is based on `count_addresses_per_asn(ipv4_single_origin)` in the BGP assignment. 

The main change is that we count the number of addresses within a prefix, not within an ASN. We call the number of addresses the weight of a given prefix. 

**TODO**: The main change is that we return two dictionaries: one is keyed by raw prefix strings, and the other is keyed by normalized prefix strings. This style of implementation has the advantage of avoiding having to parse the raw prefixes twice later on in the notebook.

In [30]:
def weigh_prefixes(pyt):
    """
    Add something later. 
    """
    def get_address_count(pfx):
        pfx_len = 32 - int(pfx.split("/")[1])
        return 1 << pfx_len

    pfx_to_weight = {}
    for pfx in pyt:
        pfx_weight = get_address_count(pfx)
        children_weight = 0
        for child in pyt.children(pfx):
            if child != pfx and pyt.parent(child) == pfx:
                children_weight += get_address_count(child)
        pfx_to_weight[pfx] = pfx_weight - children_weight
        
    return pfx_to_weight

In [34]:
def build_global_graph(RIB_PATHS):
    """
    Each element returned by bgpkit parser represents a RIB table entry with the
    following format. For example:
    {
        'type': 'R',
        'peer_asn': 1234,
        'peer_address': '80.77.16.114',
        'as-path': '1234 956 14068',
        'origin': 14068,
        'prefix': '216.163.136.0/24'
    }
    """
    vp_set = set()                     # set up viewpoints
    global_graph = defaultdict(set)    # global graph of AS paths
    pyt = pytricia.PyTricia(32)        # radix tree for prefix weighing later
    n_processed = 0                    # for reporting progress

    for p in RIB_PATHS:
        for element in bgpkit.Parser(url=str(p)):
            pfx = element.prefix
            if ":" in pfx or pfx.endswith("/0"): # skip v6 and default routes
                continue
            
            vp_set.add(element.peer_ip)
            
            # update global graph
            # (atom is an as_path excluding its origin AS)
            atom = tuple(element.as_path.split(" ")[:-1])
            global_graph[atom].add((element.origin, pfx))
            
            pyt.insert(pfx, 0)
            
            n_processed += 1
            if n_processed % 5_000_000 == 0: 
                print(f"\tPrrocessed {n_processed} entries...")
                
    print(f"Processed {n_processed} RIB entries across {len(RIB_PATHS)} snapshots.")
    return dict(global_graph), pyt

In [ ]:
global_graph, pyt = build_global_graph(RIB_PATHS)
pfx_to_weight = weigh_prefixes(pyt)

observed_addrs = sum(pfx_to_weight.values())
print(f"Weighted {len(pfx_to_weight)} prefixes.")
print(f"Observed IPv4 address space: {observed_addrs} addresses "
      f"by {len(vp_set)} viewpoints from collector(s): {', '.join(COLLECTORS)}.")

	Prrocessed 5000000 entries...
Processed 5472388 RIB entries across 1 snapshots.
Weighted 1090852 prefixes.
Observed IPv4 address space: 3120349183 addresses by 12 viewpoints from collector(s): rrc06


## Task 1: Betweenness Centrality

In this task you will:

1. Finish the implementation of the accumulation algorithm to compute $S$ and $\sum_{u, w \in V} \sigma_{uw}(v)$;
2. Write the code to compute the quotients of the accumulated values;
3. Plot a CCDF of the weighted betweenness centralities;
4. Answer some questions about the data.

In [36]:
def get_bcscore_per_asn(asn):
    pass

In [ ]:
# These three lines are included for efficiency reasons
gc.collect()
gc.freeze()
gc.disable()

t0 = time.time()

transit_u = defaultdict(int)  # AS -> path count
total_paths = 0               # Total number of paths
transit_w = defaultdict(int)  # AS -> address-weighted path count
total_weight = 0              # Total number of paths weighted by address space size
all_ases = set()              # every AS observed on a usable path, endpoints included
n_as_set = 0
route_stats = {}
for elem in iter_routes(RIB_PATHS, dedup=True, expect_routes=n_entries, stats=route_stats):
    w = weights.get(elem.prefix)

    # Filter out irrelevant paths and hops
    if w is None:  # IPv6, default route, or unparseable prefix
        continue
    path_str = elem.as_path
    if path_str is None:
        continue
    if "{" in path_str:  # AS-set: ambiguous, skip
        n_as_set += 1
        continue
    path = []
    prev = None
    for hop in path_str.split():
        if hop != prev:  # collapse prepending
            path.append(hop)
            prev = hop

    # Keep track of all seen ASes
    all_ases.update(path)

    ### STUDENT CODE START
    total_paths += 1
    total_weight += w
    for asn in set(path[1:-1]):
        transit_u[asn] += 1
        transit_w[asn] += w
    ### STUDENT CODE END

gc.enable()
gc.unfreeze()

print(f"{route_stats['announcements']:,} announcements read, "
      f"{route_stats['duplicates']:,} duplicate routes dropped "
      f"({100 * route_stats['duplicates'] / max(route_stats['announcements'], 1):.1f}%)")
print(f"{total_paths:,} distinct IPv4 paths counted ({n_as_set:,} skipped for AS-sets), "
      f"{len(all_ases):,} ASes observed, {len(transit_u):,} of them transit "
      f"({time.time() - t0:.0f}s)")

The next two cells grab AS information from RIPE and displays the centrality values computed above as a table.

In [ ]:
ASNAMES_URL = "https://ftp.ripe.net/ripe/asnames/asn.txt"
asn_info = {}
try:
    with urllib.request.urlopen(ASNAMES_URL, timeout=60) as resp:
        for line in io.TextIOWrapper(resp, encoding="utf-8", errors="replace"):
            num, _, rest = line.strip().partition(" ")
            if not num.isdigit():
                continue
            name, sep, country = rest.rpartition(", ")
            asn_info[int(num)] = (name if sep else rest, country if sep else "")
except Exception as exc:
    print(f"could not fetch AS names ({exc}); continuing with numbers only")

In [ ]:
rows = []
for asn_str in all_ases:
    n_paths = transit_u.get(asn_str, 0)
    asn_weight = transit_w.get(asn_str, 0)

    asn = int(asn_str)
    name, country = asn_info.get(asn, ("", ""))

    rows.append({
        ### STUDENT CODE START
        "bc_unweighted": n_paths / total_paths,
        "bc_weighted": asn_weight / total_weight,
        ### STUDENT CODE END

        "asn": asn,
        "name": name[:48],
        "country": country,
        "paths": n_paths,
        "addresses": asn_weight,
    })

df = (pd.DataFrame(rows)
        .sort_values("bc_weighted", ascending=False)
        .reset_index(drop=True))
df.insert(0, "rank_w", df.index + 1)
df["rank_u"] = df["bc_unweighted"].rank(ascending=False, method="min").astype(int)

df.head(25).style.format({
    "bc_weighted": "{:.4f}",
    "bc_unweighted": "{:.4f}",
    "paths": "{:,}",
    "addresses": "{:,}",
}).hide(axis="index")

### Question 1
What does betweenness centrality measure?

YOUR ANSWER HERE

### Question 2
Why would we use the weighted betweenness centrality versus the unweighted betweenness centrality?

YOUR ANSWER HERE

### Question 3
What are the top 5 ASes ranked by their weighted betweenness centralities? How these compare to the top ASes [as ranked by customer cone size](https://asrank.caida.org/)? Does this make sense? Why?

YOUR ANSWER HERE

## Distribution of centrality across ASes

The CCDF below shows, for each centrality value $x$, how many ASes have
$BC \geq x$. Edge ASes — the large majority, with $BC = 0$ — sit off the
log-log axes; what is plotted is the transit tail. Like the prefix- and
address-count distributions in the reference notebook, expect a heavy tail:
most transit ASes carry a tiny fraction of paths, while a handful of tier-1 and
large tier-2 networks each transit a sizable share of the routed address space.

In [ ]:
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"

def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_color(MUTED)
        spine.set_linewidth(0.8)
    ax.tick_params(colors=MUTED, labelcolor=INK2)
    ax.grid(True, which="both", color=GRID, linewidth=0.6, alpha=0.6)
    ax.set_axisbelow(True)

def ccdf(values):
    xs = sorted(v for v in values if v > 0)  # edge ASes (BC = 0) cannot render on log axes
    return xs, [(len(xs) - i) / len(xs) for i in range(len(xs))]

fig, ax = plt.subplots(figsize=(7, 4.5))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

xw, yw = ccdf(df["bc_weighted"])
ax.plot(xw, yw, color=BLUE, linewidth=2, label="address-weighted")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Betweenness centrality", color=INK2)
ax.set_ylabel("Proportion of ASes with BC ≥ x", color=INK2)
fig.tight_layout()
plt.show()

### Question 4
What shape does the CCDF have? What does this tell us about the distribution of weighted betweenness centralities?

YOUR ANSWER HERE

## Task: AS Hegemony

In this task you will:

1. Compute $BC_{(j)}(v)$ for each viewpoint $j$ and AS $v$;
2. Filter out VPs and compute quotients to calculate AS hegemony;
3. Create a scatter plot of AS hegemony vs. weighted betweenness centrality;
4. Answer some questions about the data.

In [ ]:
ALPHA = 0.1                # fraction of viewpoints trimmed at each end (paper's value)
FULL_FEED_FRACTION = 0.75  # a viewpoint must carry >= this fraction of all announced IPv4 prefixes

vp_threshold = FULL_FEED_FRACTION * len(norm_weight)

# peer_counts (from pass 1) sums announcements across collectors, so a peer
# feeding two collectors is counted about twice here. That only overshoots,
# never undershoots: no genuine full feed is lost at this stage, and impostors
# are re-checked against their deduplicated route count after pass 3.
vp_meta = sorted(k for k, c in peer_counts.items() if c >= vp_threshold)
vp_index = {peer_ip: j for j, (peer_ip, _) in enumerate(vp_meta)}
num_vps = len(vp_meta)

print(f"{num_vps} candidate full-feed viewpoints "
      f"(>= {vp_threshold:,.0f} of {len(norm_weight):,} IPv4 prefixes) "
      f"in {len({asn for _, asn in vp_meta})} peer ASes, "
      f"out of {len(peer_counts)} peers total")

In [ ]:
# These three lines are included for efficiency reasons
gc.collect()
gc.freeze()
gc.disable()

t0 = time.time()
n_used = 0

routes_per_vp = {}     # map from ASes and VPs to count of routes
addresses_per_vp = {}  # map from ASes and VPs to weighted count of routes
# The following three lines behave like DefaultDicts. They are more
# memory efficient for this particular task
zero_row = array("q", [0]) * num_vps  # initialize everything to 0s
vp_total_u = array("q", zero_row)     # map from VPs to count of routes
vp_total_w = array("q", zero_row)     # map from VPs to weighted count of routes

# Table to ensure each route is considered only once
seen = SeenKeys(sum(c for (ip, _), c in peer_counts.items() if ip in vp_index))

for elem in iter_routes(RIB_PATHS, dedup=False, progress_every=0):
    j = vp_index.get(elem.peer_ip)

    # Filter out VPs that see few routes
    if j is None:
        continue
    # Filter out duplicate routes
    if not seen.add(hash((elem.peer_ip, elem.prefix))):
        continue

    w = weights.get(elem.prefix)

    # Filter out IPv6, default route, or unparseable prefix
    if w is None:
        continue
    path_str = elem.as_path
    # Filter out AS-set or missing path
    if path_str is None or "{" in path_str:
        continue

    n_used += 1
    if n_used % 5_000_000 == 0:
        print(f"  {n_used:,} viewpoint routes...")

    # Accumulate weight towards the denominator of BC_j(v)
    vp_total_u[j] += 1
    vp_total_w[j] += w

    path = []
    prev = None
    for hop in path_str.split():
        if hop != prev:  # collapse prepending
            path.append(hop)
            prev = hop

    for v in path[1:-1]:
        routes_acc = routes_per_vp.get(v)
        addresses_acc = addresses_per_vp.get(v)
        if addresses_acc is None:
            # In this case, routes_acc is also None
            routes_acc = routes_per_vp[v] = array("q", zero_row)
            addresses_acc = addresses_per_vp[v] = array("q", zero_row)

        # Accumulate weight towards the numerator of BC_j(v)
        routes_acc[j] += 1
        addresses_acc[j] += w

gc.enable()
gc.unfreeze()
print(f"{n_used:,} routes from {num_vps} candidate viewpoints, "
      f"{len(addresses_per_vp):,} ASes on-path ({time.time() - t0:.0f}s)")

In [ ]:
t0 = time.time()

# We only use VPs that have a full enough view of the global Internet
valid_vp_indices = [j for j in range(num_vps) if vp_total_u[j] >= vp_threshold]
n_vp = len(valid_vp_indices)
if n_vp == 0:
    raise RuntimeError("no full-feed viewpoints — lower FULL_FEED_FRACTION or add collectors")

# Find the number (alpha proprtion) of VPs to trim
### STUDENT CODE START
n_vps_to_trim = math.floor(ALPHA * n_vp)
### STUDENT CODE END

hege_rows = []
for asn_str, vp_w in addresses_per_vp.items():
    asn = int(asn_str)

    # For each viewpoint, compute the betweenness centrality of this asn
    vp_bc_w = sorted(vp_w[j] / vp_total_w[j] for j in valid_vp_indices)
    # Trim the top and bottom alpha proportion VPs and average
    hegemony = sum(vp_bc_w[n_vps_to_trim:n_vp - n_vps_to_trim]) / (n_vp - 2 * n_vps_to_trim)

    name, country = asn_info.get(asn, ("", ""))
    hege_rows.append({
        "asn": asn,
        "name": name[:48],
        "country": country,
        "hegemony": hegemony,
        "viewpoints": sum(1 for x in vp_bc_w if x > 0),
    })

hege_df = (pd.DataFrame(hege_rows)
             .sort_values("hegemony", ascending=False)
             .reset_index(drop=True))
hege_df.insert(0, "rank_w", hege_df.index + 1)
print(f"{len(hege_df):,} ASes scored ({time.time() - t0:.0f}s)")

hege_df.head(25).style.format({
    "hegemony": "{:.4f}",
}).hide(axis="index")

### Question 1
Why do we remove the top and bottom $\alpha$ proportions of the viewpoints?

YOUR ANSWER HERE

### Question 2
How do the top 5 ASes ranked by betweenness centrality compare to the top 5 ranked by AS hegemony?

YOUR ANSWER HERE

In [ ]:
both = df.merge(hege_df, on="asn", suffixes=("_bc", "_hege"))
pos = both[(both["bc_weighted"] > 0) & (both["hegemony"] > 0)]

fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

lo = max(min(pos["bc_weighted"].min(), pos["hegemony"].min()), 1e-9)
hi = max(pos["bc_weighted"].max(), pos["hegemony"].max()) * 2
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.annotate("equal under both metrics", xy=(hi, hi),
            xytext=(-8, -14), textcoords="offset points",
            ha="right", fontsize=8, color=MUTED)

ax.scatter(pos["bc_weighted"], pos["hegemony"],
           s=14, color=BLUE, alpha=0.45, linewidths=0, zorder=2)

asn_outlier = pos.iloc[(pos["bc_weighted"] / pos["hegemony"]).argmax()]
ax.annotate(f"AS{asn_outlier['asn']}", xy=(asn_outlier["bc_weighted"], asn_outlier["hegemony"]),
            xytext=(5, 3), textcoords="offset points",
            fontsize=8, color=INK2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Weighted betweenness centrality", color=INK2)
ax.set_ylabel("AS hegemony", color=INK2)
fig.tight_layout()
plt.show()

### Question 3
What does the concentration of points on the $x = y$ line tell us about the relationship between betweenness centrality and AS hegemony?

YOUR ANSWER HERE

### Question 4
Look up the AS that corresponds to the outlier point. How does this AS's geographic location relate to the location of the collectors?

YOUR ANSWER HERE

### Question 5
Why does the geographic location of the outlier AS cause its betweenness centrality to be so much larger than its AS hegemony?

YOUR ANSWER HERE